# 02 - Preparation des donnees et Feature Engineering

**Auteur :** Gregory CRESPIN | **Date :** 30/01/2026 | **Version :** 1.0

---

DESCRIPTIF : Ce notebook prepare les donnees pour l'entrainement. On fusionne
toutes les tables, cree de nouvelles variables (feature engineering),
gere les valeurs manquantes et encode les variables categorielle.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.data_loader import load_data, merge_all_data

%matplotlib inline

## 1. Chargement et fusion des données

In [2]:
# Charger toutes les données
data_dict = load_data('../data')

# Fusionner toutes les tables
df_train, df_test = merge_all_data(data_dict)

Chargement de application_train.csv...
  Shape: (307511, 122)
Chargement de application_test.csv...
  Shape: (48744, 121)
Chargement de bureau.csv...
  Shape: (1716428, 17)
Chargement de bureau_balance.csv...
  Shape: (27299925, 3)
Chargement de previous_application.csv...
  Shape: (1670214, 37)
Chargement de POS_CASH_balance.csv...
  Shape: (10001358, 8)
Chargement de credit_card_balance.csv...
  Shape: (3840312, 23)
Chargement de installments_payments.csv...
  Shape: (13605401, 8)
Chargement de HomeCredit_columns_description.csv...
  Shape: (219, 5)
Fusion avec bureau...
Fusion avec previous_application...
Fusion avec POS_CASH_balance...
Fusion avec credit_card_balance...
Fusion avec installments_payments...

[OK] Fusion terminee
Train shape: (307511, 213)
Test shape: (48744, 212)


## 1b. Suppression des colonnes non pertinentes

DESCRIPTIF : D'apres l'exploration, certaines colonnes ont un taux de valeurs
manquantes eleve (> 60%) ET ne sont pas des donnees metiers directes pour
l'estimation du risque de defaut. Ce sont principalement les caracteristiques
du batiment (surface, etage, annee de construction...) issues de sources
externes. On les supprime pour simplifier le modele et eviter le bruit.

In [3]:
# Colonnes a supprimer : taux manquants > 60% ET non pertinentes pour le scoring
# (caracteristiques du batiment, pas de lien direct avec la capacite de remboursement)
COLS_HIGH_MISSING_LOW_RELEVANCE = [
    'COMMONAREA_MEDI', 'COMMONAREA_AVG', 'COMMONAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_MEDI',
    'FONDKAPREMONT_MODE',
    'LIVINGAPARTMENTS_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI',
    'FLOORSMIN_AVG', 'FLOORSMIN_MODE', 'FLOORSMIN_MEDI',
    'YEARS_BUILD_MEDI', 'YEARS_BUILD_MODE', 'YEARS_BUILD_AVG',
    'OWN_CAR_AGE',  # 66% manquants (clients sans voiture)
    'LANDAREA_MEDI', 'LANDAREA_MODE', 'LANDAREA_AVG',
    # Autres caracteristiques batiment (si presentes)
    'BASEMENTAREA_AVG', 'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI',
    'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI',
    'ELEVATORS_AVG', 'ELEVATORS_MODE', 'ELEVATORS_MEDI',
    'ENTRANCES_AVG', 'ENTRANCES_MODE', 'ENTRANCES_MEDI',
    'FLOORSMAX_AVG', 'FLOORSMAX_MODE', 'FLOORSMAX_MEDI',
    'LIVINGAREA_AVG', 'LIVINGAREA_MODE', 'LIVINGAREA_MEDI',
    'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MODE', 'NONLIVINGAREA_MEDI',
    'APARTMENTS_AVG', 'APARTMENTS_MODE', 'APARTMENTS_MEDI',
]

# Ne supprimer que les colonnes qui existent
cols_to_drop = [c for c in COLS_HIGH_MISSING_LOW_RELEVANCE if c in df_train.columns]
n_cols_before = len(df_train.columns)
df_train = df_train.drop(columns=cols_to_drop, errors='ignore')
df_test = df_test.drop(columns=cols_to_drop, errors='ignore')
n_cols_after = len(df_train.columns)

# Compter par difference de shape (fiable meme si cellule re-executee)
n_dropped = n_cols_before - n_cols_after
print(f"Colonnes supprimees: {n_dropped} (sur {len(cols_to_drop)} candidates trouvees)")
print(f"Train shape apres suppression: {df_train.shape}")
print(f"Test shape apres suppression: {df_test.shape}")

Colonnes supprimees: 44 (sur 44 candidates trouvees)
Train shape apres suppression: (307511, 169)
Test shape apres suppression: (48744, 168)


## 2. Feature Engineering

In [4]:
# Créer de nouvelles features
def create_features(df):
    df = df.copy()
    
    # Ratios financiers
    if 'AMT_CREDIT' in df.columns and 'AMT_INCOME_TOTAL' in df.columns:
        df['CREDIT_INCOME_PERC'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    
    if 'AMT_ANNUITY' in df.columns and 'AMT_CREDIT' in df.columns:
        df['ANNUITY_CREDIT_PERC'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    
    if 'AMT_GOODS_PRICE' in df.columns and 'AMT_CREDIT' in df.columns:
        df['GOODS_CREDIT_PERC'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']
    
    # Features temporelles
    if 'DAYS_BIRTH' in df.columns:
        df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
    
    if 'DAYS_EMPLOYED' in df.columns:
        df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365.25
        df['EMPLOYED_YEARS'] = df['EMPLOYED_YEARS'].clip(lower=0)
    
    # Features combinées
    if 'EXT_SOURCE_1' in df.columns and 'EXT_SOURCE_2' in df.columns:
        df['EXT_SOURCES_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2']].mean(axis=1)
    
    if 'EXT_SOURCE_1' in df.columns and 'EXT_SOURCE_2' in df.columns and 'EXT_SOURCE_3' in df.columns:
        df['EXT_SOURCES_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    
    return df

df_train = create_features(df_train)
df_test = create_features(df_test)

print(f"Train shape après feature engineering: {df_train.shape}")
print(f"Test shape après feature engineering: {df_test.shape}")

Train shape après feature engineering: (307511, 175)
Test shape après feature engineering: (48744, 174)


## 3. Gestion des valeurs manquantes

In [5]:
# Séparer les colonnes numériques et catégorielles
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_train.select_dtypes(include=['object']).columns.tolist()

# Retirer SK_ID_CURR et TARGET
if 'SK_ID_CURR' in numeric_cols:
    numeric_cols.remove('SK_ID_CURR')
if 'TARGET' in numeric_cols:
    numeric_cols.remove('TARGET')

print(f"Colonnes numériques: {len(numeric_cols)}")
print(f"Colonnes catégorielles: {len(categorical_cols)}")

Colonnes numériques: 158
Colonnes catégorielles: 15


In [6]:
# Imputation des valeurs manquantes numériques (médiane)
from sklearn.impute import SimpleImputer

# Remplacer les valeurs infinies par NaN (SimpleImputer ne gère pas les inf)
df_train[numeric_cols] = df_train[numeric_cols].replace([np.inf, -np.inf], np.nan)
df_test[numeric_cols] = df_test[numeric_cols].replace([np.inf, -np.inf], np.nan)

numeric_imputer = SimpleImputer(strategy='median')
df_train[numeric_cols] = numeric_imputer.fit_transform(df_train[numeric_cols])
df_test[numeric_cols] = numeric_imputer.transform(df_test[numeric_cols])

print("==> Imputation numérique terminée")

==> Imputation numérique terminée


In [7]:
# Imputation des valeurs manquantes catégorielles (mode)
if len(categorical_cols) > 0:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    df_train[categorical_cols] = categorical_imputer.fit_transform(df_train[categorical_cols])
    df_test[categorical_cols] = categorical_imputer.transform(df_test[categorical_cols])
    print("==> Imputation catégorielle terminée")
else:
    print("Aucune colonne catégorielle à imputer")

==> Imputation catégorielle terminée


## 4. Encodage des variables catégorielles

In [8]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding pour les variables catégorielles
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Combiner train et test pour avoir tous les labels
    combined = pd.concat([df_train[col], df_test[col]], axis=0)
    le.fit(combined.astype(str))
    
    df_train[col] = le.transform(df_train[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))
    
    label_encoders[col] = le

print(f"==> Encodage de {len(categorical_cols)} colonnes catégorielles terminé")

==> Encodage de 15 colonnes catégorielles terminé


## 5. Préparation finale pour l'entraînement

In [9]:
# Séparer features et target
target_col = 'TARGET'
id_col = 'SK_ID_CURR'

X_train = df_train.drop([target_col, id_col], axis=1)
y_train = df_train[target_col]

X_test = df_test.drop([id_col], axis=1)
test_ids = df_test[id_col]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nDistribution de y_train:")
print(y_train.value_counts())
print(f"\nPourcentage de classe 1: {y_train.mean() * 100:.2f}%")

X_train shape: (307511, 173)
y_train shape: (307511,)
X_test shape: (48744, 173)

Distribution de y_train:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Pourcentage de classe 1: 8.07%


In [10]:
# Sauvegarder les données préparées
X_train.to_csv('../data/X_train_prepared.csv', index=False)
y_train.to_csv('../data/y_train_prepared.csv', index=False)
X_test.to_csv('../data/X_test_prepared.csv', index=False)
test_ids.to_csv('../data/test_ids.csv', index=False)

print("=> Données sauvegardées")

=> Données sauvegardées
